In [ ]:
import cv2 as cv
import numpy as np
import time
from pynput.mouse import Controller, Button  # Kontrola myszy — pozwala pobierać i ustawiać pozycję kursora oraz wykonywać kliknięcia
import tkinter as tk  # używane do pobrania rozdzielczości ekranu

# --- Ustawienia i parametry ---
VIDEO_SOURCE = 0  # numer kamery (0 to kamera domyślna)
RESIZE_WIDTH = 640  # szerokość, do której zostanie przeskalowany obraz z kamery
V_THRESHOLD = 65  # próg jasności (dla detekcji obiektów)
MIN_AREA = 1500  # minimalna powierzchnia obiektu, żeby był uznany za wykryty
ASPECT_RATIO_MIN = 1.2  # minimalny stosunek boków prostokąta (dla obiektu)
ASPECT_RATIO_MAX = 6.0  # maksymalny stosunek boków prostokąta
APPROX_EPSILON_FACTOR = 0.02  # dokładność przybliżenia konturów (im mniejsza, tym dokładniej odwzorowuje kształt)
SAVE_COOLDOWN = 1.0  # czas między kolejnymi kliknięciami (sekundy)

# --- Ustawienia wyglądu i zachowania „strefy neutralnej” (żółty kwadrat na środku ekranu) ---
box_size = 120  # długość boku kwadratu
color_box = (0, 255, 255)  # kolor żółty w formacie BGR
thickness_box = 2  # grubość linii kwadratu
move_step = 5  # o tyle pikseli kursor się przesuwa za jednym krokiem

# --- Inicjalizacja myszy ---
mouse = Controller()  # funkcja która odpowiada za tworzenie obiektu kursora
                      # pobiera i ustawia pozycję kursora oraz pozwala na kliknięcia
last_move = time.time()  # Zapamiętuje aktualny czas (sekundy od uruchomienia systemu)
                         # Służy później do obliczania odstępu czasowego między kolejnymi ruchami kursora.
last_click_time = 0.0  # zapamiętuje czas ostatniego kliknięcia prawym przyciskiem myszy (dla ograniczenia częstotliwości)

# --- try próbuje pobrać rozdzielczość ekranu przez bibliotekę Tkinter ---
# To działa dobrze na większości komputerów, ale:
# Tkinter może być niedostępny (niezainstalowany), system może zablokować tworzenie okna (np. na serwerze bez GUI),
# mogą wystąpić błędy w odczycie ekranu (np. zdalne połączenie, inny system operacyjny).
# Gdyby któryś z tych błędów się pojawił — bez try/except program by się zatrzymał z błędem i nie działałby dalej.
try:
    root = tk.Tk()
    screen_width = root.winfo_screenwidth()   # pobiera szerokość ekranu w pikselach
    screen_height = root.winfo_screenheight() # pobiera wysokość ekranu w pikselach
    root.destroy()
# Jeśli wystąpi jakikolwiek błąd, ustawia się po prostu domyślną rozdzielczość 1920×1080
except Exception:
    screen_width, screen_height = 1920, 1080

# --- Wczytaj klasyfikator twarzy ---
face_cascade = cv.CascadeClassifier(cv.data.haarcascades + 'haarcascade_frontalface_default.xml')

# --- Uruchamia kamerę o indeksie 0, czyli domyślną kamerę podłączoną do komputera ---
# Zmienna cap to obiekt do pobierania kolejnych obrazów z kamery.
cap = cv.VideoCapture(VIDEO_SOURCE)
if not cap.isOpened():
    print("Nie można otworzyć kamery")
    exit(1)

# --- Funkcja zmieniająca rozmiar obrazu bez zniekształcania proporcji ---
def resize_keep_aspect(frame, width=None):
    if width is None:
        return frame
    h, w = frame.shape[:2]
    if w == width:
        return frame
    scale = width / float(w)
    return cv.resize(frame, (width, int(h * scale)), interpolation=cv.INTER_AREA)

# --- Pętla główna programu ---
while True:  # To nieskończona pętla – program działa, dopóki użytkownik nie przerwie.
              # Wszystko, co dzieje się w tej pętli, wykonuje się dla każdej klatki obrazu z kamery.
    ret, frame = cap.read()  # RET to wartość logiczna (True / False), czy odczyt się udał.
                             # FRAME to sama klatka (macierz obrazu).
    frame = cv.flip(frame, 1)  # Odwraca obraz poziomo (lustrzane odbicie).
                               # Dzięki temu ruch głowy w lewo/prawo jest naturalny (tak jak w lustrze).
    if not ret:
        print("Nie udało się pobrać klatki")
        break

    resized = resize_keep_aspect(frame, RESIZE_WIDTH)  # przeskalowanie obrazu dla lepszej wydajności
    orig = resized.copy()  # kopia klatki (będzie użyta do detekcji obiektów)

    # --------------------------- DETEKCJA TWARZY (sterowanie kursorem) ---------------------------
    gray = cv.cvtColor(resized, cv.COLOR_BGR2GRAY)  # konwersja obrazu na odcienie szarości
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)  # wykrywanie twarzy w obrazie gray

    height, width = resized.shape[:2]
    center_x = width // 2  # Obliczamy środek obrazu (center_x, center_y)
    center_y = height // 2
    half_box = box_size // 2  # połowa długości boku kwadratu
    # Wyznacza położenie żółtego kwadratu w centrum obrazu — tzw. strefa neutralna
    pt1 = (center_x - half_box, center_y - half_box)
    pt2 = (center_x + half_box, center_y + half_box)
    cv.rectangle(resized, pt1, pt2, color_box, thickness_box)  # rysuje kwadrat

    if len(faces) > 0:
        (x, y, w, h) = faces[0]  # bierze pierwszą wykrytą twarz
        cv.rectangle(resized, (x, y), (x+w, y+h), (255, 0, 0), 2)  # niebieska ramka wokół twarzy

        # Oblicza środek klatki i środek twarzy
        face_center_x = x + w // 2
        face_center_y = y + h // 2
        dx = face_center_x - center_x  # różnica pozioma
        dy = face_center_y - center_y  # różnica pionowa

        # Sprawdza kierunek ruchu twarzy względem środka
        direction = ""
        if dx < -30:
            direction += "Twarz w lewo "
        elif dx > 30:
            direction += "Twarz w prawo "
        if dy < -20:
            direction += "Twarz do gory "
        elif dy > 20:
            direction += "Twarz w dol "
        cv.putText(resized, direction, (20, 30), cv.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

        # Sterowanie myszką, gdy twarz poza kwadratem (czyli nie w strefie neutralnej)
        if not (pt1[0] < face_center_x < pt2[0] and pt1[1] < face_center_y < pt2[1]):
            if time.time() - last_move > 0.03:  # minimalny odstęp czasu między kolejnymi ruchami
                mouse_x, mouse_y = mouse.position  # pobiera aktualną pozycję kursora

                # Blok odpowiada za sterowanie myszką i czułość na odchylenia głowy
                if dx < -30:
                    mouse_x = max(0, mouse_x - move_step)
                elif dx > 30:
                    mouse_x = min(screen_width - 1, mouse_x + move_step)
                if dy < -20:
                    mouse_y = max(0, mouse_y - move_step)
                elif dy > 20:
                    mouse_y = min(screen_height - 1, mouse_y + move_step)

                # Ustawia nową pozycję kursora.
                mouse.position = (mouse_x, mouse_y)
                last_move = time.time()  # aktualizacja czasu ostatniego ruchu
    else:
        cv.putText(resized, "Brak wykrytej twarzy!", (20, 30), cv.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    # --------------------------- DETEKCJA OBIEKTU (kliknięcie prawym przyciskiem myszy) ---------------------------
    hsv = cv.cvtColor(orig, cv.COLOR_BGR2HSV)  # konwersja na przestrzeń barw HSV
    v = hsv[:, :, 2]  # pobranie kanału jasności (Value)
    _, mask = cv.threshold(v, V_THRESHOLD, 255, cv.THRESH_BINARY_INV)  # progowanie (ciemne obiekty stają się białe)

    # Operacje morfologiczne — usuwają szumy i wygładzają maskę
    kernel = cv.getStructuringElement(cv.MORPH_RECT, (5, 5))
    mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, kernel, iterations=2)
    mask = cv.morphologyEx(mask, cv.MORPH_OPEN, kernel, iterations=1)
    mask = cv.GaussianBlur(mask, (5, 5), 0)

    contours, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)  # wykrywa kontury obiektów
    triggered = False  # flaga wykrycia obiektu

    for cnt in contours:
        area = cv.contourArea(cnt)  # oblicza powierzchnię konturu
        if area < MIN_AREA:
            continue
        peri = cv.arcLength(cnt, True)  # obwód konturu
        approx = cv.approxPolyDP(cnt, APPROX_EPSILON_FACTOR * peri, True)  # przybliżenie do wielokąta

        if len(approx) == 4:  # interesują nas tylko prostokąty
            rect = cv.minAreaRect(cnt)  # oblicza prostokąt o minimalnym polu
            (cx, cy), (rw, rh), _ = rect
            if rw == 0 or rh == 0:
                continue
            long_side = max(rw, rh)
            short_side = min(rw, rh)
            aspect = long_side / short_side  # stosunek boków

            # Sprawdza czy prostokąt mieści się w zakresie proporcji
            if ASPECT_RATIO_MIN <= aspect <= ASPECT_RATIO_MAX:
                box = cv.boxPoints(rect)
                box = np.rint(box).astype(int)
                cv.drawContours(orig, [box], 0, (0, 255, 0), 2)  # rysuje kontur wykrytego obiektu
                triggered = True  # oznacza że wykryto obiekt

    # --- Kliknięcie prawym przyciskiem myszy przy wykryciu obiektu ---
    now = time.time()
    if triggered and (now - last_click_time) > SAVE_COOLDOWN:  # odstęp między kliknięciami
        mouse.click(Button.left, 1)  # Kliknięcie PRAWYM przyciskiem myszy
        print("Kliknięto lewym przyciskiem myszy (wykryto obiekt)")
        last_click_time = now  # zapamiętuje czas ostatniego kliknięcia

    # --------------------------- WYŚWIETLANIE OBRAZU ---------------------------
    cv.imshow("Haar Face Detect - Center Box", resized)  # okno z detekcją twarzy
    cv.imshow("Detekcja obiektu", orig)  # okno z detekcją prostokąta (obiektów)

    # Odpowiada za zamknięcie okna z kamerą klawiszem ESC
    if cv.waitKey(1) & 0xFF == 27:
        break

# --- Po zakończeniu programu ---
cap.release()  # zwalnia kamerę
cv.destroyAllWindows()  # zamyka wszystkie okna OpenCV